# 🎓 cutrug – profi lépések videóból, automatikusan

**Használat:** a táncvideókat tedd a Google Drive-on a **`cutrug tanulás / 1 - ide tedd a videókat`** mappába (a fájlnév elején lehet a stílus: pl. `House - shuffle oktató.mp4`), aztán itt: **Futtatókörnyezet → Mindegyik futtatása**.
A jegyzetfüzet minden új videót feldolgoz (GVHMR 3D mozgáskövetés – oldalra lépés, forgás, ugrás is → ütemre bontott profi nyolcasok), az eredményt **magától elküldi a Tanuló stúdióba** (`https://zorits1.github.io/zk-dance-party/?studio` – „Feldolgozott videók” lista), a videót pedig áthelyezi a `2 - feldolgozva` mappába.

**Egyszer kell:** Futtatókörnyezet → Futtatókörnyezet típusa → **T4 GPU**; ingyenes regisztráció: **https://smpl-x.is.tue.mpg.de** (a 2. cella kéri a belépést; utána a Drive-ra mentjük, legközelebb nem kell).
Csak saját vagy engedélyezett videót használj.

In [ ]:
#@title 1. Telepítés és Drive (kb. 3–6 perc; a Drive-hoz egyszer engedélyt kér)
!pip -q install "gvhmr[preproc]" librosa
!wget -q -O pro_moves.py https://raw.githubusercontent.com/zorits1/zk-dance-party/main/tools/pro_moves.py
!wget -q -O mocap_to_moves.py https://raw.githubusercontent.com/zorits1/zk-dance-party/main/tools/mocap_to_moves.py
from google.colab import drive; drive.mount('/content/drive')
import os, shutil, glob
BASE = '/content/drive/MyDrive/cutrug tanulás'; IN = BASE + '/1 - ide tedd a videókat'; DONE = BASE + '/2 - feldolgozva'; CACHE = BASE + '/.gvhmr-cache'
for d in (IN, DONE, CACHE): os.makedirs(d, exist_ok=True)
# a testmodell (SMPL-X) visszatöltése a Drive-ról – így csak egyszer kell belépni
for sub in ('.cache/gvhmr', '.gvhmr'):
    src = os.path.join(CACHE, sub.replace('/', '_'))
    if os.path.isdir(src): shutil.copytree(src, os.path.expanduser('~/' + sub), dirs_exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINCS – Futtatókörnyezet → T4 GPU!')
VIDS = sorted([p for p in glob.glob(IN + '/*') if p.lower().endswith(('.mp4', '.mov', '.mkv', '.webm', '.m4v', '.avi'))])
print(len(VIDS), 'új videó:', *[os.path.basename(v) for v in VIDS], sep='\n  ')

In [ ]:
#@title 2. Belépés a 3D testmodellhez (csak ha még nincs elmentve – a cella alatt kéri a felhasználónevet és jelszót)
import subprocess
info = subprocess.run(['gvhmr', 'info'], capture_output=True, text=True).stdout
print(info[-1500:])
if 'missing' in info.lower() or 'not found' in info.lower() or 'smplx' not in info.lower():
    !gvhmr auth smpl
for sub in ('.cache/gvhmr', '.gvhmr'):
    p = os.path.expanduser('~/' + sub)
    if os.path.isdir(p): shutil.copytree(p, os.path.join(CACHE, sub.replace('/', '_')), dirs_exist_ok=True)
print('kész')

In [ ]:
#@title 3. Feldolgozás és küldés a Tanuló stúdióba
FIREBASE = "https://zk-dance-party-default-rtdb.europe-west1.firebasedatabase.app"  #@param {type:"string"}
MAX_HOSSZ_MP = 120  #@param {type:"number"}
ALLO_KAMERA = True  #@param {type:"boolean"}
import re, json, time, hashlib, urllib.request, numpy as np, librosa, gvhmr
from mocap_to_moves import smpl_to_coco, resample, smooth, extract, STYLE_CODE
pipe = gvhmr.pipeline('human-motion-recovery', device='cuda')
def put(path, val):
    req = urllib.request.Request(f"{FIREBASE}/{path}.json", data=json.dumps(val).encode(), method='PUT', headers={'Content-Type': 'application/json'})
    urllib.request.urlopen(req, timeout=60).read()
os.makedirs('/content/work', exist_ok=True)
for v in VIDS:
    name = os.path.splitext(os.path.basename(v))[0]
    m = re.match(r'\s*([^-–]+?)\s*[-–]\s*(.+)', name); style = 'Pop'; title = name
    if m and m.group(1).strip() in STYLE_CODE: style, title = m.group(1).strip(), m.group(2).strip()
    print(f'▶ {name}  (stílus: {style})')
    try:
        clip = '/content/work/clip.mp4'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', v, '-t', str(MAX_HOSSZ_MP), '-vf', 'scale=-2:720', '-r', '30', '-c:v', 'libx264', '-c:a', 'aac', clip], check=True)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', clip, '-vn', '-ac', '1', '-ar', '22050', '/content/work/a.wav'], check=True)
        y, sr = librosa.load('/content/work/a.wav', sr=22050)
        tempo, beats = librosa.beat.beat_track(y=y, sr=sr, units='time')
        bpm = float(np.atleast_1d(tempo)[0]) or 120.0
        while bpm < 80: bpm *= 2
        while bpm > 160: bpm /= 2
        t0 = float(beats[0]) if len(beats) else None
        res = pipe(clip, static_camera=ALLO_KAMERA)
        K = smooth(resample(smpl_to_coco(np.asarray(res.joints_world)), 30.0))
        out = extract(K, bpm, style, title, '', t0=t0)
        jid = 'V' + hashlib.md5((name + str(time.time())).encode()).hexdigest()[:10]
        data = json.dumps(out, separators=(',', ':'))
        json.dump(out, open(os.path.join(DONE, name + '_cutrug.json'), 'w'), separators=(',', ':'))
        if out['moves']:
            put(f'mocap/{jid}', data)
            put(f'mocap_idx/{jid}', dict(title=title[:150], style=style, bpm=round(out['bpm'], 1), n=len(out['moves']), ts=int(time.time() * 1000)))
        shutil.move(v, os.path.join(DONE, os.path.basename(v)))
        print(f"   ✓ {len(out['moves'])} lépés, {out['bpm']} BPM → elküldve a stúdióba")
    except Exception as e:
        print('   ✗ hiba:', e)
print('Kész. Nyisd meg: https://zorits1.github.io/zk-dance-party/?studio  → „Feldolgozott videók”')